# 📊 RAG Evaluation & Metrics

This notebook demonstrates pgVectorDB's built-in **RAG evaluator** for measuring retrieval quality.

### Metrics Computed
| Metric | What It Measures |
|:-------|:-----------------|
| **Precision@K** | Fraction of retrieved docs that are relevant |
| **Recall@K** | Fraction of all relevant docs that were retrieved |
| **F1@K** | Harmonic mean of precision and recall |
| **MAP** | Mean Average Precision (rank-aware) |
| **MRR** | Mean Reciprocal Rank (position of first relevant doc) |
| **NDCG@K** | Normalized Discounted Cumulative Gain |
| **Hit Rate** | Fraction of queries with ≥1 relevant result |

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from pgvectordb import RAGEvaluator

## 1. Basic Evaluation

Evaluate retrieval results against ground truth.

In [ ]:
evaluator = RAGEvaluator(k=5)

# Simulated data
queries = [
    "What is vector search?",
    "How does BM25 work?",
    "Best database for AI?",
]

# What your system actually returned (doc IDs)
retrieved = [
    ["doc_1", "doc_3", "doc_7", "doc_2", "doc_5"],
    ["doc_4", "doc_6", "doc_1", "doc_8", "doc_2"],
    ["doc_2", "doc_1", "doc_9", "doc_3", "doc_4"],
]

# What SHOULD have been returned (ground truth)
ground_truth = [
    ["doc_1", "doc_2", "doc_3"],  # 3 relevant docs
    ["doc_4", "doc_6"],  # 2 relevant docs
    ["doc_2", "doc_9", "doc_10"],  # 3 relevant docs
]

result = evaluator.evaluate(queries, retrieved, ground_truth)
print(result)

## 2. Per-Query Drill-Down

In [ ]:
for i, query in enumerate(queries):
    metrics = evaluator.evaluate_single_query(
        retrieved_docs=retrieved[i],
        relevant_docs=ground_truth[i],
    )
    print(f"\n📝 Query: '{query}'")
    print(f"   Retrieved:    {retrieved[i]}")
    print(f"   Ground truth: {ground_truth[i]}")
    for metric, value in metrics.items():
        print(f"   {metric}: {value:.3f}")

## 3. Export Results

In [ ]:
result_dict = result.to_dict()
print("📋 Exportable dict:")
for k, v in result_dict.items():
    print(f"  {k}: {v:.4f}")

## 4. Comparing K Values

See how metrics change at different K values.

In [ ]:
print(
    f"{'K':>3} | {'Precision':>10} | {'Recall':>8} | {'F1':>6} | {'MRR':>6} | {'NDCG':>6}"
)
print("-" * 55)

for k_val in [1, 2, 3, 5, 10]:
    ev = RAGEvaluator(k=k_val)
    # Pad retrieved lists to at least k_val
    padded = [r + [f"pad_{j}" for j in range(k_val)] for r in retrieved]
    res = ev.evaluate(queries, padded, ground_truth)
    print(
        f"{k_val:>3} | {res.precision:>10.3f} | {res.recall:>8.3f} | {res.f1_score:>6.3f} | {res.mrr_score:>6.3f} | {res.ndcg_score:>6.3f}"
    )

## 5. Live Recall Measurement

With a live database, use `compute_recall()` to measure ANN vs exact search quality.

```python
# Requires a running pgVectorDB instance:
recall = await rag.compute_recall(
    test_queries=["vector search", "database index"],
    k=10,
)
print(f"Recall@10: {recall['recall@k']:.2%}")
```

In [ ]:
print("✅ Evaluation complete!")